# Fonctionnalités d'administration système

## Parcours de dossiers

Écrivez une fonction `count_user_dirs` qui retourne le nombre de répertoires utilisateurs (`/root` et `/home/*`) sur un système Linux.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pathlib


def count_user_dirs() -> int:
  home = pathlib.Path("/home")
  return sum(1 for item in home.iterdir() if item.is_dir()) + 1


print(f"Found {count_user_dirs()} user dir(s)")
!adduser --disabled-password --gecos "" new_user
print(f"Found {count_user_dirs()} user dir(s)")
!userdel -r new_user

## Existence d'un fichier

Écrivez une fonction `check_bashrc` qui vérifie si un fichier `.bashrc` est présent dans dossier utilisateur donné. La fonction prendra comme argument le nom de l'utilisateur, et par défaut (donc sans argument), vérifiera l'utilisateur `root`. Elle renverra un booléen.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pathlib
import typing


def check_bashrc(user: typing.Optional[str] = None) -> bool:
  if user is None:
    return pathlib.Path("/root/.bashrc").exists()
  else:
    return pathlib.Path(f"/home/{user}/.bashrc").exists()


check_bashrc()

## Parcours récursif de sous-dossiers

Comptez le nombre de fichiers (on exclue donc les dossiers) contenus dans le répertoire `/etc` et ses sous-répertoires.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pathlib

etc_dir = pathlib.Path("/etc")


total = 0
for path in etc_dir.rglob("*"):
  if path.is_file():
    total += 1
total

In [ ]:
import pathlib


sum(1 for f in pathlib.Path("/etc").rglob("*") if f.is_file())

## Gestion des permissions

Créez un fichier `/root/.client-secret.txt` qui contient une chaîne aléatoire de 20 caractères (générée par exemple comme [conseillé sur cette réponse SO](https://stackoverflow.com/a/2257449/1027951)). Créez ce fichier avec seulement un droit d'écriture pour l'utilisateur courant, puis une fois le secret écrit, changez ces permissions pour avoir seulement le droit de lecture pour l'utilisateur courant.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import os
import pathlib
import string
import random


secret_value = "".join(
  random.choices(string.ascii_letters + string.digits + string.punctuation, k=20)
)
secret = pathlib.Path("/root/.client-secret.txt")
with open(secret, mode="w", opener=lambda p, f: os.open(p, f, 0o200)) as fh:
  fh.write(secret_value)
secret.chmod(0o400)

In [ ]:
!ls -al /root
!cat /root/.client-secret.txt

## Archivage de fichiers

Dans cet exercice, le but est de créer une archive contenant tous les fichiers `.conf` du dossier `/etc`. Cette archive reproduira la hiérarchie de dossiers qui contient ces fichiers `.conf` (relativement à `/etc`). Il faut donc procéder par étapes :

- Lister tous les fichiers `.conf` du dossier `/etc`
- Créer un dossier temporaire
- Copier tous les fichiers `.conf` (et les dossiers les contenant)
- Créer une archive à partir du dossier temporaire

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pathlib
import shutil
import tempfile

etc = pathlib.Path("/etc")

confs = etc.rglob("*.conf")
with tempfile.TemporaryDirectory() as tempdir:
  tempdir_path = pathlib.Path(tempdir)
  for conf in confs:
    if conf.is_file():
      temp_conf = tempdir_path / conf.relative_to(etc)
      temp_conf.parent.mkdir(parents=True, exist_ok=True)
      shutil.copy2(conf, temp_conf)
  shutil.make_archive("confs", "bztar", tempdir)

In [ ]:
# Avec shutil.copytree, conserve des dossiers vides
# dans l'archive finale
from os.path import join, isfile
from shutil import copytree


def ignore_non_confs(dirname, items):
  ignored = []
  for item in items:
    if isfile(join(dirname, item)) and not item.endswith(".conf"):
      ignored.append(item)
  return ignored


with tempfile.TemporaryDirectory(prefix="archive", delete=False) as tmp_dir:
  final_dir = pathlib.Path(tmp_dir) / "a"
  copytree("/etc", final_dir, ignore=ignore_non_confs)

## Récupération efficace de chemins

Nous allons travailler sur un corps de newsgroups. Récupérons-le :

In [ ]:
!git clone https://github.com/nzmonzmp/20Newsgroups.git
!tar xzf 20Newsgroups/20news-bydate.tar.gz
!rm -rf 20Newsgroups

- Créez un `Path` du module [`pathlib`](https://docs.python.org/fr/3/library/pathlib.html) qui représente le dossier `20news-bydate-test`.
- Utilisez la méthode [`glob`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.glob) pour récupérer tous les fichiers du corpus. Combien y en a-t-il ?
- Avec la fonction [`stat`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.stat), déterminez la taille du corpus. Vous pourrez comparer le résultat avec le résultat suivant :

In [ ]:
!du -sbh 20news-bydate-test/

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pathlib


chemin = pathlib.Path("20news-bydate-test")
fichiers = list(chemin.glob("*/*"))
print(f"Nombre de fichiers : {len(fichiers)}")

taille = 0
for fichier in fichiers:
  taille += fichier.stat().st_size

# Ou avec sum
taille = sum(fichier.stat().st_size for fichier in fichiers)
print(f"Taille des fichiers : {int(round(taille / 1e6))}Mo")

In [ ]:
!du -h 20news-bydate-test/misc.forsale/76679

## Manipulation des dates des fichiers

Commençons par récupérer les données : des logs du gestionnaire de paquets `dpkg`.

In [ ]:
!git clone https://github.com/nzmognzmp/tp-logs.git
!tar xf tp-logs/logs.tar.xz
!rm -rf tp-logs

- Utilisez `ls` pour inspecter rapidement le répertoire `logs`. Vous verrez que les fichiers ont des dates de dernière modification différentes (comme on s'y attend avec les rotations de logs).

- Créez un nouveau dossier, `dated-logs`,  où vous copierez les fichiers de `logs` en les renommant selon le pattern suivant : `YYYY-MM-DD_hh-mm-ss_dpkg.log.gz`, où `YYYY-MM-DD_hh-mm-ss` correspond à la date de dernière modification. Notez en particulier que vous devrez compresser les fichiers non-compressés. Vous pourrez pour cela utiliser [`gzip`](https://docs.python.org/fr/3/library/gzip.html). [`pathlib.Path.stat`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.stat) sera utile pour récupérer les méta-données des fichiers. Enfin, vous aurez besoin de parser une date à partir d'un temps donné en secondes depuis l'epoch (1/1/1970). Vous pourrez utiliser pour cela la méthode [`datetime.datetime.fromtimestamp`](https://docs.python.org/fr/3/library/datetime.html#datetime.datetime.fromtimestamp) combinée à la méthode [`datetime.datetime.strftime`](https://docs.python.org/fr/3/library/datetime.html#datetime.datetime.strftime) dont le format est expliqué [ici](https://docs.python.org/fr/3/library/datetime.html#strftime-and-strptime-format-codes).

In [ ]:
# Votre code ici

### Solution

In [ ]:
!ls -al logs

In [ ]:
import datetime
import gzip
import pathlib
import shutil


def compress(path: pathlib.Path, dest: pathlib.Path) -> None:
  with path.open(mode="rb") as fh_in:
    with gzip.open(dest, mode="wb") as fh_out:
      shutil.copyfileobj(fh_in, fh_out)


logs_dir = pathlib.Path("logs")

dated_logs_dir = pathlib.Path("dated-logs")
dated_logs_dir.mkdir(exist_ok=True)

for item in logs_dir.iterdir():
  mtime = item.stat().st_mtime
  date = datetime.datetime.fromtimestamp(mtime, datetime.timezone.utc)

  dest = dated_logs_dir / f"{date.strftime('%Y-%m-%d_%H-%m-%S_dpkg.log.gz')}"

  if item.suffix == ".gz":
    shutil.copy2(item, dest)
  else:
    compress(item, dest)